In [ ]:
# file modified from original Apache-2.0 licensed code from https://github.com/Understanding-Visual-Datasets/VisDiff
# see LICENSE and NOTICE files in the root directory for details


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
if os.getcwd().split(os.sep)[-1] == 'notebooks':
    os.chdir('..') # set/ ccwd as the parent directory to make imports easier

import pandas as pd
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid

In [ ]:
df = pd.read_csv('data/PairedImageSets.csv')
# # randomly pick a difference
diff_choice = df.sample(1).difference.values[0]
df = df[df['difference'] == diff_choice]
df.head()


In [ ]:
(group_a, group_b) = df['group_name'].unique()
group_a = df[df['group_name'] == group_a].path.values[:10]
group_b = df[df['group_name'] == group_b].path.values[:10]

fig = plt.figure(figsize=(30., 10.))
# set fig title
# fig.suptitle(diff_choice, fontsize=20)
print(diff_choice)
grid = ImageGrid(fig, 111,  # similar to subplot(111)
                 nrows_ncols=(2, 10),  # creates 2x2 grid of axes
                 axes_pad=0.03,  # pad between axes in inch.
                 )

for ax, im in zip(grid, list(group_a) + list(group_b)):
    # Iterating over the grid returns the Axes.
    ax.imshow(Image.open(im).convert("RGB").resize((224, 224)))
    ax.axis('off')

In [ ]:
df['group_name'].unique()

In [ ]:
# interactive browsing of all difference groups

from IPython.display import clear_output
import time
import pandas as pd
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid

# load full dataset (don't overwrite the filtered `df` already in the notebook)
full_df = pd.read_csv('data/PairedImageSets.csv')

# iterate with index so we can go forward/backward
diffs = list(full_df['difference'].unique())
idx = 0
while 0 <= idx < len(diffs):
    diff = diffs[idx]
    sub = full_df[full_df['difference'] == diff]
    group_names = sub['group_name'].unique()
    if len(group_names) != 2:
        # skip unexpected groups by advancing index
        idx += 1
        continue
    ga_name, gb_name = group_names

    # --- visualization: 10 randomly sampled images from first 100 of each set ---
    # take up to first 100 images from each group, then sample up to 10 without replacement
    all_a = sub[sub['group_name'] == ga_name].path.values[:100]
    all_b = sub[sub['group_name'] == gb_name].path.values[:100]

    # sample
    rng = np.random.default_rng()
    sample_a = []
    sample_b = []
    if len(all_a) > 0:
        k_a = min(10, len(all_a))
        sample_a = list(rng.choice(all_a, size=k_a, replace=False))
    if len(all_b) > 0:
        k_b = min(10, len(all_b))
        sample_b = list(rng.choice(all_b, size=k_b, replace=False))

    fig = plt.figure(figsize=(30., 10.))
    fig.suptitle(f"{ga_name}\n\n{gb_name}", fontsize=34, fontweight='bold')
    grid = ImageGrid(fig, 111, nrows_ncols=(2, 10), axes_pad=0.03)

    samp_imgs = sample_a + sample_b
    for ax, im_path in zip(grid, samp_imgs):
        ax.imshow(Image.open(im_path).convert("RGB").resize((224, 224)))
        ax.axis('off')
    for ax in grid[len(samp_imgs):]:
        ax.axis('off')

    plt.show()
    # ------------------------------------------------------------------------------

    # wait for user action: Enter=next, b=back, q=quit
    try:
        resp = input("Press Enter for next pair, 'b' to go back, or 'q' to quit: ").strip().lower()
    except KeyboardInterrupt:
        resp = 'q'
    if resp == 'q':
        clear_output(wait=True)
        print("Stopped by user.")
        break
    elif resp == 'b':
        if idx > 0:
            idx -= 1
        else:
            print("Already at the first pair; cannot go back.")
        clear_output(wait=True)
        continue
    else:
        idx += 1
        clear_output(wait=True)